In [0]:
# %sql
# DROP TABLE IF EXISTS silver.dimvertical;

In [0]:
# 1. Reads bronze.workertable.
# 2. Extracts distinct Vertical values.
# 3. Creates silver.dimvertical with auto-generated VerticalId.
# 4. Compares source verticals against existing silver.dimvertical.
# 5. Inserts only new verticals.
# 6. Lets Delta generate VerticalId automatically.

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimvertical"

###Read Bronze tables

In [0]:
# Read Bronze worker table
workerdf = spark.table("bronze.workertable")
display(workerdf)

###Create Silver Dimension Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.dimvertical (
        VerticalId BIGINT GENERATED ALWAYS AS IDENTITY,
        Vertical STRING
)
USING DELTA

###Build Dimension Table

In [0]:
df = workerdf.select(F.expr("trim(Vertical) AS Vertical")).distinct()
display(df)

In [0]:
verticaldf = spark.table("silver.dimvertical")
display(verticaldf)

In [0]:
newrowsdf = df.filter(F.col("Vertical").isNotNull()).exceptAll(verticaldf.select("Vertical"))
display(newrowsdf)

In [0]:
spark.sql("insert into silver.dimvertical(Vertical) select Vertical from {newrowsdf}", newrowsdf=newrowsdf)


In [0]:
display(spark.table("silver.dimvertical"))